# IAgilize | Gestão Inteligente de Demandas

**Prova de Conceito (POC) acadêmica** para demonstrar como IA generativa pode interpretar mensagens de clientes e organizar solicitações de profissionais autônomos, com foco em Social Media.

## Como executar no Google Colab
1. Abra este notebook no Google Colab.
2. Em **Secrets** (ícone de chave na barra lateral), crie `GEMINI_API_KEY` com uma chave válida do Google AI Studio e habilite o acesso para o notebook.
3. Execute a célula de instalação abaixo.
4. Execute a célula principal e teste uma das mensagens de exemplo.

> A chave não deve ser escrita no notebook nem enviada ao GitHub. Esta POC não envia mensagens reais por WhatsApp e mantém os registros apenas em memória durante a sessão.

In [2]:
%pip install -q google-genai gradio


## Implementação


In [4]:
import json
import gradio as gr
from google import genai
from google.genai import types

# No Colab, cadastre GEMINI_API_KEY em Secrets (ícone de chave na barra lateral).
# Fora do Colab, defina a variável de ambiente GEMINI_API_KEY.
def obter_chave_api():
    try:
        from google.colab import userdata
        chave = userdata.get("GEMINI_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    import os
    return os.getenv("GEMINI_API_KEY")

chave_api = obter_chave_api()
client = genai.Client(api_key=chave_api) if chave_api else None

# Dados mantidos apenas em memória durante a execução da demonstração.
banco_dados = {
    "tarefas": [],
    "financeiro": {"receitas_previstas": 0.0}
}

def agente_nlp_extrator(mensagem_cliente: str) -> dict:
    """Interpreta a mensagem e extrai informações conforme o catálogo de demonstração."""
    if not client:
        raise RuntimeError(
            "Chave Gemini não configurada. Adicione GEMINI_API_KEY em Colab > Secrets."
        )

    prompt = f"""
Você é a inteligência analítica do IAgilize, uma demonstração para profissionais de Social Media.
Interprete a mensagem usando exclusivamente o catálogo abaixo.

CATÁLOGO DE REFERÊNCIA
Planos mensais:
- Básico: R$ 600
- Intermediário: R$ 1.200
- Avançado: R$ 1.800
- Premium: R$ 2.500
- Completo: R$ 3.500

Extras:
- Post avulso: R$ 50 cada
- Reel: R$ 80 cada
- Consultoria: R$ 150 cada

Retorne um JSON com:
- cliente: nome ou "Não identificado"
- plano_escolhido: plano ou "Nenhum/Apenas Extras"
- demanda: resumo do pedido
- prazo: prazo mencionado ou "A combinar"
- valor_total: número. Se for apenas consulta/orçamento sem contratação, use 0.

Mensagem do cliente:
{mensagem_cliente}
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.1
        )
    )
    dados = json.loads(response.text)

    campos = ["cliente", "plano_escolhido", "demanda", "prazo", "valor_total"]
    if not all(campo in dados for campo in campos):
        raise ValueError("A resposta da IA não contém todos os campos esperados.")
    return dados

def processar_fluxo_chatbot(mensagem_usuario, historico):
    try:
        dados = agente_nlp_extrator(mensagem_usuario)
    except Exception as erro:
        return (
            "Não consegui processar a mensagem nesta tentativa. "
            "Verifique a configuração da chave Gemini e tente novamente.\n\n"
            f"Detalhe técnico: `{type(erro).__name__}`"
        )

    cliente = str(dados.get("cliente", "Não identificado"))
    demanda = str(dados.get("demanda", "Não identificada"))
    plano = str(dados.get("plano_escolhido", "Nenhum"))
    prazo = str(dados.get("prazo", "A combinar"))

    try:
        valor = max(0.0, float(dados.get("valor_total", 0.0)))
    except (ValueError, TypeError):
        valor = 0.0

    palavras_duvida = [
        "quanto custa", "orçamento", "orcamento", "preco",
        "preço", "valores", "gostaria de saber"
    ]
    texto = mensagem_usuario.lower()
    eh_apenas_orcamento = valor == 0 or (
        any(p in texto for p in palavras_duvida)
        and "fechar" not in texto
        and "quero" not in texto
    )

    if eh_apenas_orcamento:
        status_demanda = "Cotação / Interesse de Cliente"
        status_financeiro = "Em negociação. Nenhum valor somado às receitas previstas."
        notificacao_cliente = (
            f"Olá, {cliente}! Recebemos seu interesse em: {demanda}. "
            "Vamos conversar para definir os detalhes."
        )
    else:
        status_demanda = "Contrato Fechado (Plano Ativo)"
        status_financeiro = f"R$ {valor:.2f} adicionados às receitas previstas (simulação)."
        banco_dados["financeiro"]["receitas_previstas"] += valor
        notificacao_cliente = (
            f"Perfeito, {cliente}! Registramos a contratação de {demanda} "
            f"por R$ {valor:.2f}. Início/prazo: {prazo}."
        )

    nova_tarefa = {
        "id": len(banco_dados["tarefas"]) + 1,
        "descricao": demanda,
        "tipo": status_demanda,
        "cliente": cliente,
        "prazo": prazo,
        "valor": valor
    }
    banco_dados["tarefas"].append(nova_tarefa)

    return f"""## IAgilize: mensagem processada

### Painel do profissional
- **Cliente:** {cliente}
- **Plano identificado:** {plano}
- **Demanda:** {demanda}
- **Prazo:** {prazo}
- **Tarefa registrada:** #{nova_tarefa['id']} ({status_demanda})
- **Financeiro:** {status_financeiro}

### Indicadores da demonstração
- **Demandas na sessão:** {len(banco_dados['tarefas'])}
- **Receitas previstas na sessão:** R$ {banco_dados['financeiro']['receitas_previstas']:.2f}

### Resposta sugerida ao cliente
{notificacao_cliente}

---
*Demonstração acadêmica: o registro é mantido em memória e a mensagem não é enviada por WhatsApp.*
"""

interface = gr.ChatInterface(
    fn=processar_fluxo_chatbot,
    title="IAgilize | Gestão Inteligente",
    description=(
        "POC acadêmica para interpretar solicitações de clientes de Social Media "
        "e organizar demandas em uma demonstração de painel."
    ),
    examples=[
        "Olá, meu nome é Roberta. Gostaria de fechar o plano Intermediário para começar na próxima segunda!",
        "Oi, aqui é o Bruno da Hamburgueria. Quero fechar o plano Básico e adicionar 2 Reels extras. Qual o total?",
        "Boa tarde, sou a Camila. Quanto custa o plano Avançado?"
    ]
)

if __name__ == "__main__":
    interface.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://31bdd18be3d4eb4f0d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://31bdd18be3d4eb4f0d.gradio.live
